# Legacy mixed follow-up validity check

This notebook checks whether the mixed rank + expert follow-up report is valid enough to use for conclusions. The Excel report is the source of truth for selected benchmark rows; source artifacts on the noctua2 cluster are used to verify checkpoints, final eval loss, target modules, training data, runtime, and training exposure.

Source report:
`outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/mix_follow_up_report.xlsx`

First we define criteria for what each run should fulfill to be considered "valid".

Validity criteria:
- all approaches train on the same data consisting of 10 languages
- all approaches train for approx. 7.6h
- benchmark result file exists and has all expected benchmark-family rows
- selected adapter/checkpoint artifact exists and is the adapter checkpoint used for benchmark evaluation
- final eval loss is present when claimed
- LoRA target modules are full, or explicitly intended for MoV / documented as reduced follow-up configuration
- actual training-data exposure is checked explicitly; runs with `epoch > 1` are kept as a validity caveat


In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("/scratch/hpc-prf-merlin/joel/moe-study")

NOTEBOOK_DIR = ROOT / "result_analysis/notebooks/validation"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from util import (
    adapter_dir_from_result,
    as_path,
    load_training_metrics,
    norm_targets,
    read_json,
    result_path_kind,
    run_dir,
    selected_checkpoint_step,
    source_root_from_result,
)

REPORT = ROOT / "outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/mix_follow_up_report.xlsx"
CONSOLIDATED_ROOT = ROOT / "outputs/pipeline_runs/moe_study_mixed_rank_expert_followup_consolidated_current_report"
BENCHMARK_SUMMARY = CONSOLIDATED_ROOT / "benchmarks_materialized_latest/benchmark_summary.json"
SUMMARY = CONSOLIDATED_ROOT / "summary_refreshed.csv"

FULL_TARGETS = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
MOV_TARGETS = {"k_proj", "v_proj", "down_proj"}
ATTENTION_TARGETS = {"q_proj", "k_proj", "v_proj", "o_proj"}

print(REPORT, REPORT.exists())
print(BENCHMARK_SUMMARY, BENCHMARK_SUMMARY.exists())
print(SUMMARY, SUMMARY.exists())


/scratch/hpc-prf-merlin/joel/moe-study/outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/mix_follow_up_report.xlsx True
/scratch/hpc-prf-merlin/joel/moe-study/outputs/pipeline_runs/moe_study_mixed_rank_expert_followup_consolidated_current_report/benchmarks_materialized_latest/benchmark_summary.json True
/scratch/hpc-prf-merlin/joel/moe-study/outputs/pipeline_runs/moe_study_mixed_rank_expert_followup_consolidated_current_report/summary_refreshed.csv True


## Validation logic

Generic artifact helpers live in `util.py`. The validation-specific logic stays here so the criteria remain visible in the notebook. The mixed report does not have a `source_manifest` sheet, so the selected source root and evaluated checkpoint are reconstructed from the consolidated profile and `benchmarks_materialized_latest/benchmark_summary.json`.


In [2]:
raw = pd.read_excel(REPORT, sheet_name="benchmark_raw")
configs = pd.read_excel(REPORT, sheet_name="run_configs")
summary = pd.read_csv(SUMMARY) if SUMMARY.exists() else pd.DataFrame()
profile = read_json(CONSOLIDATED_ROOT / "state/resolved_profile.json")
benchmark_summary = read_json(BENCHMARK_SUMMARY)

non_baseline = raw[~raw["is_baseline"].fillna(False).astype(bool)].copy()
expected_raw_rows = int(non_baseline.groupby("run_id").size().max())
expected_families = sorted(non_baseline["benchmark"].dropna().astype(str).unique())

primary_mask = (
    ((non_baseline.benchmark == "Belebele") & (non_baseline.metric == "acc_norm")) |
    ((non_baseline.benchmark == "FLORES") & (non_baseline.metric.astype(str).str.startswith("bits_per_byte "))) |
    ((non_baseline.benchmark == "SIB200") & (non_baseline.metric == "acc")) |
    ((non_baseline.benchmark == "Taxi1500") & (non_baseline.metric == "acc"))
)
expected_primary_rows = int(non_baseline[primary_mask].groupby("run_id").size().max())

run_sources = profile.get("consolidated_report_sources", {}).get("run_sources", {})
benchmark_by_run = {str(row.get("run_id")): row for row in benchmark_summary.get("runs", [])}
summary_by_run = {str(row.run_id): row._asdict() for row in summary.itertuples(index=False)} if not summary.empty else {}
cfg_by_run = {str(row.run_id): row._asdict() for row in configs.itertuples(index=False)}


def source_root_for(run_id):
    source_name = run_sources.get(str(run_id), "")
    return ROOT / "outputs/pipeline_runs" / source_name if source_name else None


def final_eval_for(run_id, rd):
    path = rd / "final_eval_loss/eval_results.json" if rd else None
    payload = read_json(path) if path else {}
    return {
        "final_eval_present": bool(payload),
        "final_eval_loss": payload.get("eval_loss", summary_by_run.get(str(run_id), {}).get("eval_loss")),
        "final_eval_source_path": str(path) if path and path.exists() else "",
    }


def target_policy(kind, targets):
    target_set = set(targets)
    if not targets:
        return "unknown", "target modules unavailable"
    if target_set == FULL_TARGETS:
        return "full", "full q/k/v/o + gate/up/down target set"
    if kind == "movlora" and target_set == MOV_TARGETS:
        return "intended_mov_core", "intended MoV core target set: k_proj,v_proj,down_proj"
    if target_set == ATTENTION_TARGETS:
        return "reduced", "attention-only q/k/v/o follow-up configuration"
    return "reduced", "non-full target set"


def validity_notes(row):
    notes = []
    for col, note in [
        ("source_root_exists", "source_root_missing"),
        ("result_exists", "benchmark_result_missing"),
        ("benchmark_complete", "benchmark_incomplete"),
        ("primary_complete", "primary_metrics_incomplete"),
        ("adapter_exists", "adapter_missing"),
        ("adapter_config_exists", "adapter_config_missing"),
        ("final_eval_present", "final_eval_loss_missing"),
    ]:
        if not bool(row.get(col)):
            notes.append(note)
    if row.get("target_module_policy") == "reduced":
        notes.append("reduced_target_modules")
    if bool(row.get("actual_repeat")):
        notes.append("actual_repeat_over_one_epoch")
    return ";".join(notes)


def audit_mixed():
    rows = []
    for run_id, group in sorted(non_baseline.groupby("run_id")):
        kind = str(group["kind"].iloc[0])
        bm = benchmark_by_run.get(str(run_id), {})
        result_path = as_path(bm.get("source_result_file"), ROOT)
        source_root = source_root_for(run_id) or source_root_from_result(result_path)
        rd = run_dir(source_root, str(run_id))
        adapter_dir = None
        if source_root and bm.get("evaluated_checkpoint"):
            adapter_dir = source_root / "runs" / str(run_id) / str(bm.get("evaluated_checkpoint"))
        if adapter_dir is None or not adapter_dir.exists():
            adapter_dir = adapter_dir_from_result(result_path, source_root, str(run_id))
        adapter_cfg = read_json(adapter_dir / "adapter_config.json") if adapter_dir else {}
        cfg_row = cfg_by_run.get(str(run_id), {})
        targets = norm_targets(adapter_cfg.get("target_modules")) or norm_targets(cfg_row.get("target_modules"))
        policy, reason = target_policy(kind, targets)
        family_counts = group.groupby("benchmark").size().to_dict()
        raw_rows = int(len(group))
        primary_rows = int(len(group[primary_mask.loc[group.index]]))
        row = {
            "run_id": str(run_id),
            "kind": kind,
            "source_root": str(source_root) if source_root else "",
            "source_profile": source_root.name if source_root else "",
            "source_root_exists": bool(source_root and source_root.exists()),
            "result_path": str(result_path) if result_path else "",
            "result_path_kind": result_path_kind(bm.get("source_result_file")),
            "result_exists": bool(result_path and result_path.exists()),
            "evaluated_checkpoint": bm.get("evaluated_checkpoint", ""),
            "num_tasks_seen": bm.get("num_tasks_seen"),
            "num_tasks_expected": bm.get("num_tasks_expected"),
            "run_dir": str(rd) if rd else "",
            "adapter_dir": str(adapter_dir) if adapter_dir else "",
            "adapter_exists": bool(adapter_dir and adapter_dir.exists()),
            "adapter_config_exists": bool(adapter_cfg),
            "selected_checkpoint_step": selected_checkpoint_step(adapter_dir),
            "benchmark_rows": raw_rows,
            "expected_benchmark_rows": expected_raw_rows,
            "primary_rows": primary_rows,
            "expected_primary_rows": expected_primary_rows,
            "benchmark_complete": raw_rows == expected_raw_rows and all(fam in family_counts for fam in expected_families),
            "primary_complete": primary_rows == expected_primary_rows,
            "benchmark_family_counts": json.dumps(family_counts, sort_keys=True),
            "target_modules": ",".join(targets),
            "target_module_policy": policy,
            "target_module_reason": reason,
            "max_steps_reported": cfg_row.get("max_steps"),
        }
        row.update(final_eval_for(run_id, rd))
        row.update(load_training_metrics(rd))
        rows.append(row)
    out = pd.DataFrame(rows)
    for col in ["epoch", "num_train_epochs", "global_step", "max_steps", "max_steps_reported", "num_input_tokens_seen", "train_runtime"]:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    out["actual_repeat"] = out["epoch"] > 1.0001
    out["configured_repeat_risk"] = out["num_train_epochs"] > 1
    out["validity_notes"] = out.apply(validity_notes, axis=1)
    out["validity_status"] = np.where(out["validity_notes"].eq(""), "valid", "valid_with_note")
    return out


### Overview of mixed follow-up runs


In [3]:
runs = audit_mixed()

overview = pd.DataFrame([{
    "selected_runs": runs.run_id.nunique(),
    "complete_benchmarks": int(runs.benchmark_complete.sum()),
    "complete_primary_metrics": int(runs.primary_complete.sum()),
    "final_eval_present": int(runs.final_eval_present.sum()),
    "actual_repeat": int(runs.actual_repeat.sum()),
    "full_targets": int((runs.target_module_policy == "full").sum()),
    "intended_mov_core": int((runs.target_module_policy == "intended_mov_core").sum()),
    "reduced_targets": int((runs.target_module_policy == "reduced").sum()),
}])

display(overview)
display(runs[["run_id", "kind", "source_profile", "evaluated_checkpoint", "validity_status", "validity_notes"]].sort_values("run_id"))


,selected_runs,complete_benchmarks,complete_primary_metrics,final_eval_present,actual_repeat,full_targets,intended_mov_core,reduced_targets
0,14,14,14,14,7,6,1,7


,run_id,kind,source_profile,evaluated_checkpoint,validity_status,validity_notes
0,hydralora_r2_heads2,hydralora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-1921_adapter,valid,
1,hydralora_r2_heads32,hydralora,moe_study_mixed_rank_expert_followup_central_n...,checkpoint-896_adapter,valid,
2,hydralora_r4_heads2,hydralora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-3049_adapter,valid_with_note,actual_repeat_over_one_epoch
3,lora_r4,lora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-5219_adapter,valid_with_note,actual_repeat_over_one_epoch
4,moelora_tr128_e32_k4,moelora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-1953_adapter,valid_with_note,reduced_target_modules
5,moelora_tr128_e32_k8,moelora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-1431_adapter,valid_with_note,reduced_target_modules
6,moelora_tr4_e2_k1,moelora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-4426_adapter,valid_with_note,reduced_target_modules;actual_repeat_over_one_...
7,moelora_tr64_e32_k4,moelora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-1977_adapter,valid_with_note,reduced_target_modules
8,moelora_tr64_e32_k8,moelora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-1578_adapter,valid_with_note,reduced_target_modules
9,moelora_tr8_e2_k1,moelora,moe_study_mixed_rank_expert_followup_safe12_no...,checkpoint-4437_adapter,valid_with_note,reduced_target_modules;actual_repeat_over_one_...


### First validation check: data


In [4]:
def infer_tokenized_path(source_root):
    profile = read_json(Path(source_root) / "state/resolved_profile.json")
    configured = profile.get("tokenization", {}).get("output_dir")
    if configured:
        path = Path(configured)
        return path if path.is_absolute() else ROOT / path
    return ROOT / "outputs/pipeline_runs/moe_study_profile_otus_cluster_10_langs/tokenized"


def tokenized_meta(path):
    check = read_json(Path(path) / "_check.json")
    train_tokens = sum(check.get("tokens_per_language_by_split", {}).get("train", {}).values())
    return pd.Series({
        "tokenized_path": str(Path(path).relative_to(ROOT)),
        "num_language_ids": check.get("num_languages"),
        "languages": ",".join(sorted(check.get("samples_per_language", {}).keys())),
        "train_rows": check.get("splits", {}).get("train"),
        "train_tokens": train_tokens,
    })


data_check = runs[["kind", "run_id", "source_root", "source_profile"]].copy()
data_check["tokenized_abs"] = data_check.source_root.map(infer_tokenized_path)
data_check = pd.concat([data_check, data_check.tokenized_abs.apply(tokenized_meta)], axis=1)
display(data_check.groupby(["tokenized_path", "num_language_ids", "languages", "train_rows", "train_tokens"]).agg(
    runs=("run_id", "nunique"), kinds=("kind", lambda s: ",".join(sorted(set(s))))
).reset_index())

same_language_set = data_check.languages.nunique() == 1
same_exact_dataset = data_check.tokenized_path.nunique() == 1
display(Markdown(f"**Training data verdict:** same language/script set = `{same_language_set}`; exact same tokenized dataset path = `{same_exact_dataset}`. The metadata has `{data_check.num_language_ids.iloc[0]}` language/script IDs from the 10-language profile."))


,tokenized_path,num_language_ids,languages,train_rows,train_tokens,runs,kinds
0,outputs/pipeline_runs/moe_study_profile_otus_c...,13,"arb_Arab,arb_Latn,azj_Cyrl,azj_Latn,cmn_Hani,e...",99410,101795840,14,"hydralora,lora,moelora,mola,movlora"


**Training data verdict:** same language/script set = `True`; exact same tokenized dataset path = `True`. The metadata has `13` language/script IDs from the 10-language profile.

All mixed follow-up approaches use the same 10-language tokenized dataset. We have 13 language IDs because some languages are represented with different scripts.


### Next check: same training time / compute budget

This is important since the mixed follow-up compares selected configurations under the same longer compute budget.


In [5]:
runtime = runs.copy()
runtime["train_hours"] = runtime.train_runtime / 3600
runtime_summary = runtime.groupby(["kind"]).agg(
    runs=("run_id", "nunique"),
    hours_min=("train_hours", "min"),
    hours_median=("train_hours", "median"),
    hours_max=("train_hours", "max"),
).reset_index()
display(runtime_summary)

within_approx_76h = runtime.train_hours.between(7.25, 7.75).all()
display(Markdown(f"**Runtime verdict:** all selected runs are within 7.25-7.75h = `{within_approx_76h}`; median runtime is `{runtime.train_hours.median():.2f}h`."))


,kind,runs,hours_min,hours_median,hours_max
0,hydralora,3,7.404099,7.408496,7.509326
1,lora,1,7.539885,7.539885,7.539885
2,moelora,6,7.414051,7.509636,7.582227
3,mola,3,7.489769,7.514700,7.586986
4,movlora,1,7.456094,7.456094,7.456094


**Runtime verdict:** all selected runs are within 7.25-7.75h = `True`; median runtime is `7.51h`.

The small variations are created due to checkpoint materialization and different required time for checkpoint saving, which can depend on the compute node, approach, and FSDP config. The intended budget was 7.6h for the mixed follow-up runs.


## Benchmark and final evaluation coverage

This checks if we have all benchmark metric results for FLORES / Belebele / SIB / Taxi for all selected runs. The significance notebook uses the 42 primary metric rows per run; the report also contains additional raw benchmark rows.

Final eval loss is a separate post-training stage. It is not used to decide which approach is better, but it is useful as an artifact health check.


In [6]:
coverage = runs.groupby(["kind"]).agg(
    runs=("run_id", "nunique"),
    benchmark_complete=("benchmark_complete", "sum"),
    primary_complete=("primary_complete", "sum"),
    final_eval_present=("final_eval_present", "sum"),
    result_path_kinds=("result_path_kind", lambda s: ",".join(sorted(set(s)))),
).reset_index()
display(coverage)

display(runs[[
    "run_id", "kind", "benchmark_rows", "expected_benchmark_rows", "primary_rows", "expected_primary_rows", "num_tasks_seen", "final_eval_present", "final_eval_loss"
]].sort_values("run_id"))


,kind,runs,benchmark_complete,primary_complete,final_eval_present,result_path_kinds
0,hydralora,3,3,3,3,file
1,lora,1,1,1,1,file
2,moelora,6,6,6,6,file
3,mola,3,3,3,3,file
4,movlora,1,1,1,1,file


,run_id,kind,benchmark_rows,expected_benchmark_rows,primary_rows,expected_primary_rows,num_tasks_seen,final_eval_present,final_eval_loss
0,hydralora_r2_heads2,hydralora,93,93,42,42,42,True,2.337214
1,hydralora_r2_heads32,hydralora,93,93,42,42,42,True,14.738133
2,hydralora_r4_heads2,hydralora,93,93,42,42,42,True,2.337582
3,lora_r4,lora,93,93,42,42,42,True,2.276720
4,moelora_tr128_e32_k4,moelora,93,93,42,42,42,True,2.320843
5,moelora_tr128_e32_k8,moelora,93,93,42,42,42,True,2.377785
6,moelora_tr4_e2_k1,moelora,93,93,42,42,42,True,2.320843
7,moelora_tr64_e32_k4,moelora,93,93,42,42,42,True,2.320843
8,moelora_tr64_e32_k8,moelora,93,93,42,42,42,True,2.377785
9,moelora_tr8_e2_k1,moelora,93,93,42,42,42,True,2.320843


### LoRA target module checks

We use this check because some follow-up configurations use reduced target modules. Full target modules are marked as `full`. MoV uses `k_proj,v_proj,down_proj` by design, so this run is marked as `intended_mov_core`. Other non-full target sets are real configuration caveats and are grouped below by reason.


In [7]:
target_counts = runs.groupby(["target_module_policy"]).size().reset_index(name="runs")
display(target_counts)

reduced_summary = runs[runs.target_module_policy.eq("reduced")].groupby(["kind", "target_modules", "target_module_reason"]).agg(
    runs=("run_id", "nunique"),
    run_ids=("run_id", lambda s: ", ".join(sorted(s))),
).reset_index()
display(reduced_summary)

display(runs[runs.target_module_policy.eq("intended_mov_core")][["run_id", "kind", "target_modules"]])


,target_module_policy,runs
0,full,6
1,intended_mov_core,1
2,reduced,7


,kind,target_modules,target_module_reason,runs,run_ids
0,moelora,"k_proj,o_proj,q_proj,v_proj",attention-only q/k/v/o follow-up configuration,6,"moelora_tr128_e32_k4, moelora_tr128_e32_k8, mo..."
1,mola,"k_proj,o_proj,q_proj,v_proj",attention-only q/k/v/o follow-up configuration,1,mola_tr64_e32_k4


,run_id,kind,target_modules
13,mov_dense_e2,movlora,"down_proj,k_proj,v_proj"


## Training exposure

This checks whether a run saw the data more than once, e.g. because the data was not enough for the longer compute budget and training repeated epoch data.


In [8]:
exposure = runs.groupby(["kind", "configured_repeat_risk", "actual_repeat"]).agg(
    runs=("run_id", "nunique"),
    epoch_min=("epoch", "min"),
    epoch_median=("epoch", "median"),
    epoch_max=("epoch", "max"),
    global_step_min=("global_step", "min"),
    global_step_max=("global_step", "max"),
    max_steps_values=("max_steps", lambda s: ",".join(map(str, sorted(set(s.dropna().astype(int)))))),
).reset_index()
display(exposure)

display(runs[[
    "run_id", "kind", "num_train_epochs", "epoch", "actual_repeat", "global_step", "max_steps", "num_input_tokens_seen", "source_profile"
]].sort_values(["actual_repeat", "kind", "run_id"], ascending=[False, True, True]))


,kind,configured_repeat_risk,actual_repeat,runs,epoch_min,epoch_median,epoch_max,global_step_min,global_step_max,max_steps_values
0,hydralora,True,False,2,0.432606,0.680070,0.927534,896,1921,"10000,20000"
1,hydralora,True,True,1,1.471715,1.471715,1.471715,3049,3049,20000
2,lora,True,True,1,2.518989,2.518989,2.518989,5219,5219,20000
3,moelora,True,False,4,0.690943,0.852433,0.954534,1431,1977,20000
4,moelora,True,True,2,2.136155,2.138811,2.141466,4426,4437,20000
5,mola,True,False,1,0.508248,0.508248,0.508248,1579,1579,20000
6,mola,True,True,2,1.792275,1.792637,1.792999,2475,2476,20000
7,movlora,True,True,1,3.295367,3.295367,3.295367,1707,1707,20000


,run_id,kind,num_train_epochs,epoch,actual_repeat,global_step,max_steps,num_input_tokens_seen,source_profile
2,hydralora_r4_heads2,hydralora,10,1.471715,True,3049,20000,149823488,moe_study_mixed_rank_expert_followup_safe12_no...
3,lora_r4,lora,10,2.518989,True,5219,20000,256458752,moe_study_mixed_rank_expert_followup_safe12_no...
6,moelora_tr4_e2_k1,moelora,10,2.136155,True,4426,20000,217464832,moe_study_mixed_rank_expert_followup_safe12_no...
9,moelora_tr8_e2_k1,moelora,10,2.141466,True,4437,20000,218005504,moe_study_mixed_rank_expert_followup_safe12_no...
10,mola_tr4_e2_k1,mola,15,1.792999,True,2476,20000,182538240,moe_study_mixed_rank_expert_followup_central_n...
12,mola_tr8_e2_k1,mola,15,1.792275,True,2475,20000,182464512,moe_study_mixed_rank_expert_followup_central_n...
13,mov_dense_e2,movlora,39,3.295367,True,1707,20000,335609856,moe_study_mixed_rank_expert_followup_central_n...
0,hydralora_r2_heads2,hydralora,10,0.927534,False,1921,20000,94420992,moe_study_mixed_rank_expert_followup_safe12_no...
1,hydralora_r2_heads32,hydralora,5,0.432606,False,896,10000,44040192,moe_study_mixed_rank_expert_followup_central_n...
4,moelora_tr128_e32_k4,moelora,10,0.942947,False,1953,20000,95993856,moe_study_mixed_rank_expert_followup_safe12_no...


This shows the main mixed-follow-up caveat: several faster configurations crossed `epoch > 1` under the 7.6h compute budget. They still ran under the same wall-clock budget and same tokenized dataset, but they did not all see the same number of unique training examples before repetition.


In [9]:
missing_eval = runs.run_id.nunique() - int(runs.final_eval_present.sum())
reduced = int((runs.target_module_policy == "reduced").sum())
mov_core = int((runs.target_module_policy == "intended_mov_core").sum())
actual_repeat = int(runs.actual_repeat.sum())
complete = int(runs.benchmark_complete.sum())
primary_complete = int(runs.primary_complete.sum())

lines = [
    "## Validity conclusion",
    "",
    f"- Mixed follow-up: {complete}/{runs.run_id.nunique()} selected runs have complete benchmark rows; {primary_complete}/{runs.run_id.nunique()} have the 42 primary rows used by the significance notebook.",
    f"- Runtime: all selected runs are within the 7.25-7.75h window = `{runs.train_runtime.div(3600).between(7.25, 7.75).all()}`.",
    f"- Training data: all selected runs use the same 10-language tokenized dataset = `{data_check.tokenized_path.nunique() == 1}`.",
    f"- Final eval loss caveat: missing for {missing_eval} runs. It is not used for winner selection.",
    f"- Target-module caveat: {reduced} runs use reduced non-MoV target modules; {mov_core} MoV run uses the intended core target set (`k_proj,v_proj,down_proj`).",
    f"- Training-exposure caveat: {actual_repeat} runs crossed `epoch > 1`, so the mixed follow-up is an equal wall-clock comparison but not a strict equal-unique-token exposure comparison.",
    "",
    "Overall: the mixed follow-up report is usable for benchmark-based conclusions, but the conclusion should mention the two main caveats: repeated-data exposure for faster runs and reduced target modules for some configurations.",
]
display(Markdown("\n".join(lines)))


## Validity conclusion

- Mixed follow-up: 14/14 selected runs have complete benchmark rows; 14/14 have the 42 primary rows used by the significance notebook.
- Runtime: all selected runs are within the 7.25-7.75h window = `True`.
- Training data: all selected runs use the same 10-language tokenized dataset = `True`.
- Final eval loss caveat: missing for 0 runs. It is not used for winner selection.
- Target-module caveat: 7 runs use reduced non-MoV target modules; 1 MoV run uses the intended core target set (`k_proj,v_proj,down_proj`).
- Training-exposure caveat: 7 runs crossed `epoch > 1`, so the mixed follow-up is an equal wall-clock comparison but not a strict equal-unique-token exposure comparison.

Overall: the mixed follow-up report is usable for benchmark-based conclusions, but the conclusion should mention the two main caveats: repeated-data exposure for faster runs and reduced target modules for some configurations.